# Ablation Studies

Two ablations on target Ω₂ = {11, 11.2, 13}:
1. **Encoding base comparison** — bases 1–5, 3-qubit circuit, 1 FM per qubit
2. **Circuit architecture scaling** — ternary init, varying qubit count and FM count

Produces `fig:ablation`.

In [ ]:
# ── Smoke test flag ────────────────────────────────────────────
SMOKE_TEST = False

from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import optax
import pennylane as qml
import pandas as pd
import pickle
import sys
from sklearn.metrics import r2_score

sys.path.append("..")
from paper_style import apply_paper_style, BLUE, RED, GREEN, ORANGE, PURPLE
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

jax.config.update("jax_enable_x64", True)
apply_paper_style()
SEED = 42
np.random.seed(SEED)

## Hyperparameters

In [ ]:
NUM_FUNCTIONS  = 1 if SMOKE_TEST else 10
NUM_RUNS       = 2 if SMOKE_TEST else 10
MAX_STEPS      = 50  if SMOKE_TEST else 5000
LR             = 0.001
BATCH_SIZE     = 40
NUM_SERIAL_LAYERS  = 2
TRAINABLE_BLOCKS   = 1
NUM_SU_GATES       = 1

DATA_DIR = Path("../datasets")
OUT_DIR  = Path("results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"SMOKE_TEST={SMOKE_TEST}  functions={NUM_FUNCTIONS}  "
      f"runs={NUM_RUNS}  steps={MAX_STEPS}")

## Circuit and training helpers

In [ ]:
def S(alpha, x, num_wires, enc_layer):
    for w in range(num_wires):
        qml.RX(alpha[w] * x, wires=w)

def W_SU(theta, trainable_block_layers, num_wires, wires=None):
    if wires is None:
        wires = range(num_wires)
    for i in range(trainable_block_layers):
        qml.SpecialUnitary(theta[i][0], wires=list(wires))

def weights_ones(n_serial, t_blocks, n_su, n_params):
    return np.ones((n_serial, t_blocks, n_su, n_params))

def square_loss(targets, predictions):
    return 0.5 * sum((t-p)**2 for t,p in zip(targets, predictions)) / len(targets)


def run_experiment(initial_alpha, num_wires, num_rot_params,
                   datasets, label, num_functions, num_runs,
                   max_steps, lr, batch_size):
    """Run Adam training for a given circuit config and return a results DataFrame."""
    dev = qml.device("default.qubit", wires=num_wires)

    @qml.qnode(dev, interface="jax")
    def circuit(alpha, weights_SU, x=None):
        W_SU(weights_SU[0], TRAINABLE_BLOCKS, num_wires)
        for i in range(NUM_SERIAL_LAYERS - 1):
            S(alpha[i], x, num_wires, i)
            W_SU(weights_SU[i+1], TRAINABLE_BLOCKS, num_wires)
        return qml.expval(qml.PauliZ(wires=num_wires-1))

    circuit_jit = jax.jit(circuit)

    def cost_fn(p, x, y):
        preds = [circuit_jit(p["alpha"], p["weights_SU"].items(), x_) for x_ in x]
        return square_loss(y, preds).squeeze()

    @jax.jit
    def update(p, s, x, y):
        loss, g = jax.value_and_grad(cost_fn)(p, x, y)
        u, s2   = opt.update(g, s, p)
        return optax.apply_updates(p, u), s2, loss

    results = []
    for func in range(num_functions):
        ds = datasets[func]
        x_tr, y_tr = ds["x_train"], ds["y_train"]
        x_te, y_te = ds["x_test"],  ds["y_test"]

        for run in range(num_runs):
            alpha   = jnp.array(initial_alpha)
            weights = jnp.array(weights_ones(
                NUM_SERIAL_LAYERS, TRAINABLE_BLOCKS,
                NUM_SU_GATES, num_rot_params
            ))
            params    = {"weights_SU": weights, "alpha": alpha}
            opt       = optax.adam(lr)
            opt_state = opt.init(params)

            for step in range(max_steps):
                idx = np.random.randint(0, len(x_tr), batch_size)
                params, opt_state, c = update(
                    params, opt_state,
                    jnp.array(x_tr[idx]), jnp.array(y_tr[idx])
                )

            preds = [float(circuit_jit(params["alpha"],
                                        params["weights_SU"], x_))
                     for x_ in x_te]
            r2 = r2_score(y_te, preds)
            print(f"  {label}  func={func+1}  run={run+1}  R²={r2:.4f}")

            results.append({
                "Approach":    label,
                "Func":        func,
                "Run":         run,
                "R2_Score":    r2,
                "final_alpha": params["alpha"].tolist()
            })

    return pd.DataFrame(results)

## Load data

In [ ]:
with open(DATA_DIR / "datasets_1d_jaderberg_10.pkl", "rb") as f:
    datasets = pickle.load(f)
print(f"Loaded {len(datasets)} functions (Ω₂)")

## Ablation 1: Encoding base (3 qubits, 1 FM per qubit)

In [ ]:
# Each config: (label, initial_alpha, num_rot_params)
# Prefactors: base^0, base^1, base^2 for 3 qubits
BASE_CONFIGS = [
    ("Unary (base-1)",      [[1.0,  1.0,  1.0 ]], 63),
    ("Binary (base-2)",     [[1.0,  2.0,  4.0 ]], 63),
    ("Ternary (base-3)",    [[1.0,  3.0,  9.0 ]], 63),
    ("Quaternary (base-4)", [[1.0,  4.0,  16.0]], 63),
    ("Quinary (base-5)",    [[1.0,  5.0,  25.0]], 63),
]

base_dfs = []
for label, alpha_init, n_rot in BASE_CONFIGS:
    print(f"\n── {label} ──")
    df = run_experiment(
        alpha_init, num_wires=3, num_rot_params=n_rot,
        datasets=datasets, label=label,
        num_functions=NUM_FUNCTIONS, num_runs=NUM_RUNS,
        max_steps=MAX_STEPS, lr=LR, batch_size=BATCH_SIZE
    )
    base_dfs.append(df)

base_df = pd.concat(base_dfs, ignore_index=True)
base_df.to_csv(OUT_DIR / "ablation_base.csv", index=False)
print("\nBase ablation saved.")

## Ablation 2: Circuit architecture (ternary init)

In [ ]:
# Each config: (label, initial_alpha, num_wires, num_rot_params)
# num_rot_params: SU(2^n) has 4^n - 1 params
ARCH_CONFIGS = [
    ("2-qubit (1 FM)",  [[1.0, 3.0]],             2, 15),   # SU(4)
    ("3-qubit (1 FM)",  [[1.0, 3.0, 9.0]],         3, 63),   # SU(8)
    ("4-qubit (1 FM)",  [[1.0, 3.0, 9.0, 27.0]],   4, 255),  # SU(16)
    # 2-qubit with 2 FMs per qubit — 4 prefactors, 2 serial layers
    # Note: this requires num_serial_ansatz_layers=3 (2 FM layers)
    # Handled separately below
]

arch_dfs = []
for label, alpha_init, n_wires, n_rot in ARCH_CONFIGS:
    print(f"\n── {label} ──")
    df = run_experiment(
        alpha_init, num_wires=n_wires, num_rot_params=n_rot,
        datasets=datasets, label=label,
        num_functions=NUM_FUNCTIONS, num_runs=NUM_RUNS,
        max_steps=MAX_STEPS, lr=LR, batch_size=BATCH_SIZE
    )
    arch_dfs.append(df)

In [ ]:
# 2-qubit 2FM — separate because it uses 4 serial FM layers
# alpha shape: [[alpha_q0_layer0, alpha_q1_layer0],
#               [alpha_q0_layer1, alpha_q1_layer1]]
# i.e. {1,3} for layer 0, {9,27} for layer 1
LABEL_2Q2FM  = "2-qubit (2 FM)"
ALPHA_2Q2FM  = [[1.0, 3.0], [9.0, 27.0]]  # 2 serial FM layers
N_SERIAL_2FM = 3   # 2 FM layers → 3 ansatz blocks
N_ROT_2FM    = 15  # SU(4)

dev2 = qml.device("default.qubit", wires=2)

@qml.qnode(dev2, interface="jax")
def circuit_2q2fm(alpha, weights_SU, x=None):
    W_SU(weights_SU[0], TRAINABLE_BLOCKS, 2)
    for i in range(N_SERIAL_2FM - 1):
        S(alpha[i], x, 2, i)
        W_SU(weights_SU[i+1], TRAINABLE_BLOCKS, 2)
    return qml.expval(qml.PauliZ(wires=1))

circuit_2q2fm_jit = jax.jit(circuit_2q2fm)

results_2q2fm = []
for func in range(NUM_FUNCTIONS):
    ds = datasets[func]
    x_tr, y_tr = ds["x_train"], ds["y_train"]
    x_te, y_te = ds["x_test"],  ds["y_test"]

    for run in range(NUM_RUNS):
        alpha   = jnp.array(ALPHA_2Q2FM)
        weights = jnp.array(weights_ones(
            N_SERIAL_2FM, TRAINABLE_BLOCKS, NUM_SU_GATES, N_ROT_2FM
        ))
        params    = {"weights_SU": weights, "alpha": alpha}
        opt       = optax.adam(LR)
        opt_state = opt.init(params)

        def cost_2q2fm(p, x, y):
            preds = [circuit_2q2fm_jit(p["alpha"], p["weights_SU"], x_)
                     for x_ in x]
            return square_loss(y, preds).squeeze()

        @jax.jit
        def update_2q2fm(p, s, x, y):
            loss, g = jax.value_and_grad(cost_2q2fm)(p, x, y)
            u, s2   = opt.update(g, s, p)
            return optax.apply_updates(p, u), s2, loss

        for step in range(MAX_STEPS):
            idx = np.random.randint(0, len(x_tr), BATCH_SIZE)
            params, opt_state, c = update_2q2fm(
                params, opt_state,
                jnp.array(x_tr[idx]), jnp.array(y_tr[idx])
            )

        preds = [float(circuit_2q2fm_jit(params["alpha"],
                                          params["weights_SU"].items(), x_))
                 for x_ in x_te]
        r2 = r2_score(y_te, preds)
        print(f"  {LABEL_2Q2FM}  func={func+1}  run={run+1}  R²={r2:.4f}")

        results_2q2fm.append({
            "Approach":    LABEL_2Q2FM,
            "Func":        func,
            "Run":         run,
            "R2_Score":    r2,
            "final_alpha": params["alpha"].tolist()
        })

arch_dfs.append(pd.DataFrame(results_2q2fm))
arch_df = pd.concat(arch_dfs, ignore_index=True)
arch_df.to_csv(OUT_DIR / "ablation_arch.csv", index=False)
print("Architecture ablation saved.")

## Plot — fig:ablation

In [ ]:
from paper_style import boxplot_panel

base_df = pd.read_csv(OUT_DIR / "ablation_base.csv")
arch_df = pd.read_csv(OUT_DIR / "ablation_arch.csv")

# Panel (a) — base comparison
base_labels = ["Unary (base-1)", "Binary (base-2)", "Ternary (base-3)",
               "Quaternary (base-4)", "Quinary (base-5)"]
db = {l: base_df[base_df.Approach==l].R2_Score.values for l in base_labels}
colors_base = [RED, RED, GREEN, ORANGE, RED]

# Panel (b) — architecture scaling
arch_labels = ["2-qubit (1 FM)", "3-qubit (1 FM)",
               "4-qubit (1 FM)", "2-qubit (2 FM)"]
da = {l: arch_df[arch_df.Approach==l].R2_Score.values for l in arch_labels}
colors_arch = [RED, GREEN, BLUE, GREEN]

fig, axes = plt.subplots(1, 2, figsize=(6.8, 3.2),
                         gridspec_kw={"width_ratios": [5, 4]})
plt.subplots_adjust(wspace=0.15)

boxplot_panel(axes[0], db, colors_base, show_ylabel=True)
boxplot_panel(axes[1], da, colors_arch, show_ylabel=False)

# Shared legend
patches = [
    mpatches.Patch(fc=GREEN,  alpha=0.70, ec="black", lw=0.8,
                   label=r"$100\%$ success (provable coverage)"),
    mpatches.Patch(fc=ORANGE, alpha=0.70, ec="black", lw=0.8,
                   label=r"$100\%$ success (coincidental)"),
    mpatches.Patch(fc=BLUE,   alpha=0.70, ec="black", lw=0.8,
                   label=r"Partial success"),
    mpatches.Patch(fc=RED,    alpha=0.70, ec="black", lw=0.8,
                   label=r"Complete failure"),
]
fig.legend(handles=patches, loc="lower center", ncol=2,
           frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.03))

plt.savefig("ablation_base_and_qubits.pdf", dpi=600, bbox_inches="tight")
plt.savefig("ablation_base_and_qubits.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")

## Summary statistics

In [ ]:
for name, d in [("Base comparison", db), ("Architecture", da)]:
    print(f"\n{name}:")
    print(f"  {'Condition':<22} {'N':>4} {'Median':>7} {'Success%':>9}")
    for lbl, vals in d.items():
        print(f"  {lbl:<22} {len(vals):>4} {np.median(vals):>7.3f} "
              f"{np.mean(vals>=0.95)*100:>8.0f}%")